## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [5]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None
        self.value = value

    @property
    def value(self):
        print("__value by @property")
        return self.__value

    @value.setter
    def value(self, new_value):
        print("__value by @value.setter")
        self.__value = new_value

obj = Encapsulated(10)
print(obj.value)  # Accessing via property
print(obj._Encapsulated__value)  # Accessing via name mangling

__value by @value.setter
__value by @property
10
10


## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [18]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass

class Dog(Animal):
    def sound(self):
        return "Woof!"

class Bird(Animal):
    def sound(self):
        return "Chirp!"

try:
    animal = Animal()
except Exception as e:
    print(e)

dog = Dog()
bird = Bird()
print(dog.sound())
print(bird.sound())
print(isinstance(dog, Animal))
print(issubclass(Dog, Animal))

Can't instantiate abstract class Animal without an implementation for abstract method 'sound'
Woof!
Chirp!
True
True


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [40]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> None:
        ...

class Circle:
    def draw(self):
        print("Drawing circle")

class Square:
    def draw(self):
        print("Drawing square")

def paint(obj: Drawable):
    obj.draw()

paint(Circle())
paint(Square())

Drawing circle
Drawing square


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [39]:
class Logger:
    def __getattribute__(self, name):
        print(f"__getattribute__: {name}")
        return object.__getattribute__(self, name)

    def __getattr__(self, name):
        print(f"__getattr__: {name} not found")
        return None

obj = Logger()
obj.x = 5
print(obj.x)
print(obj.y)

__getattribute__: x
5
__getattribute__: y
__getattr__: y not found
None


## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [38]:
class Meta(type):
    def __new__(mcs, name, bases, namespace):
        print(f"Creating class: {name}")
        namespace['added_by_meta'] = True
        return super().__new__(mcs, name, bases, namespace)

class GoodClass(metaclass=Meta):
    pass

print(GoodClass.added_by_meta)

Creating class: GoodClass
True


## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [37]:
class DataDescriptor:
    def __init__(self):
        self.data = {}
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.data.get(id(instance), None)
    
    def __set__(self, instance, value):
        print(f"DataDescriptor.__set__: {value}")
        self.data[id(instance)] = value

class NonDataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "NonDataDescriptor value"

class MyClass2:
    data_desc = DataDescriptor()
    non_data_desc = NonDataDescriptor()

obj = MyClass2()
obj.data_desc = 10
print(obj.data_desc)
print(obj.non_data_desc)

DataDescriptor.__set__: 10
10
NonDataDescriptor value


## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [36]:
class A:
    def method(self):
        print("A.method")

class B(A):
    def method(self):
        print("B.method")
        super().method()

class C(A):
    def method(self):
        print("C.method")
        super().method()

class D(B, C):
    def method(self):
        print("D.method")
        super().method()

d = D()
d.method()
print(D.__mro__)

D.method
B.method
C.method
A.method
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [35]:
class LifeCycle:
    def __new__(cls, *args, **kwargs):
        print(f"__new__ called")
        return super().__new__(cls)
    
    def __init__(self, name="obj"):
        print(f"__init__ called: {name}")
        self.name = name
    
    def __repr__(self):
        return f"LifeCycle('{self.name}')"
    
    def __str__(self):
        return f"LifeCycle: {self.name}"
    
    def __call__(self):
        print(f"__call__ invoked on {self.name}")
    
    def __del__(self):
        print(f"__del__ called for {self.name}")

obj = LifeCycle("test")
print(obj)
obj()
del obj

__new__ called
__init__ called: test
LifeCycle: test
__call__ invoked on test
__del__ called for test


## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [33]:
class Access:
    def __init__(self):
        self._secret = "Secret Data"
        self._access = False
    
    @property
    def secret(self):
        if self._access:
            return self._secret
        return "Access Denied"
    
    def __enter__(self):
        print("== Granting access ==")
        self._access = True
        return self._secret
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("== Revoking access ==")
        self._access = False
        return False

obj = Access()
print(obj.secret)
with obj as secret_value:
    print(secret_value)
print(obj.secret)

Access Denied
== Granting access ==
Secret Data
== Revoking access ==
Access Denied


## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [45]:
import weakref
import gc

class Tracked:
    _instances = []
    
    def __init__(self, name):
        self.name = name
        weak_ref = weakref.ref(self, self._on_delete)
        self._instances.append(weak_ref)
        print(f"Created {name}, alive: {self.count_alive()}")
    
    @classmethod
    def _on_delete(cls, ref):
        cls._instances.remove(ref)
    
    @classmethod
    def count_alive(cls):
        cls._instances = [r for r in cls._instances if r() is not None]
        return len(cls._instances)
    
    def __del__(self):
        print(f"Deleting {self.name}")

obj1 = Tracked("A")
obj2 = Tracked("B")
obj3 = Tracked("C")
print(f"Alive: {Tracked.count_alive()}")

del obj2
gc.collect()
print(f"After del obj2: {Tracked.count_alive()}")

Created A, alive: 1
Deleting A
Created B, alive: 2
Created C, alive: 3
Deleting C
Alive: 3
Deleting B
After del obj2: 2
